In [4]:
import os
import streamlit as st
import requests
from langchain_core.tools import tool
from currentsapi import CurrentsAPI
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["OPENWEATHER_API_KEY"] = os.getenv("OPENWEATHER_API_KEY")
os.environ["CURRENTS_API_KEY"] = os.getenv("CURRENTS_API_KEY")

In [5]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo-1106",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
output_parser = StrOutputParser()

In [6]:
# Tool 1: Get latitude and longitude
@tool
def get_city_lat_lon(city: str, country_code: str) -> tuple:
    """
    Retrieves the latitude and longitude of a specified city.
    """
    api_key = os.environ["OPENWEATHER_API_KEY"]
    url = f"http://api.openweathermap.org/geo/1.0/direct?q={city},{country_code}&limit=1&appid={api_key}&mode=JSON"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        return data[0]["lat"], data[0]["lon"]
    else:
        return None, None

# Tool 2: Get Weather Information
@tool
def get_weather(lat: float, lon: float) -> str:
    """
    Retrieves the current weather information for the given latitude and longitude.
    """
    api_key = os.environ["OPENWEATHER_API_KEY"]
    url = f"https://api.openweathermap.org/data/2.5/weather?lat={lat}&lon={lon}&appid={api_key}&units=metric"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        description = data['weather'][0]['description']
        temperature = data['main']['temp']
        humidity = data['main']['humidity']
        wind_speed = data['wind']['speed']
        return (f"Weather: {description}\n"
                f"Temperature: {temperature} ℃\n"
                f"Humidity: {humidity}%\n"
                f"Wind Speed: {wind_speed} m/s")
    else:
        return "Unable to retrieve weather data."

# Tool 3: Get Current News
@tool
def get_current_news(keyword: str) -> str:
    """
    Fetches the latest news articles based on the provided keyword.
    """
    currents = CurrentsAPI(api_key=os.environ["CURRENTS_API_KEY"])
    response = currents.search(keywords=keyword)

    if response['status'] == 'ok':
        news_list = []
        for article in response['news']:
            news_item = f"Title: {article['title']}\nDescription: {article['description']}\nURL: {article['url']}\nPublished: {article['published']}\n{'-' * 50}"
            news_list.append(news_item)
        return "\n".join(news_list)
    else:
        return "Failed to fetch news."

In [7]:
tools = [get_city_lat_lon, get_weather,get_current_news]
llm_with_tools = llm.bind_tools(tools)

In [8]:
# Define the system message that sets the context for the chatbot
system_message = """
You are a helpful and knowledgeable travel assistant named TravelSynk. Your role is to assist users in planning their travel itineraries by providing accurate and relevant information. You have access to the following tools:

1. **get_city_lat_lon**: Retrieves the latitude and longitude of a specified city.
2. **get_weather**: Retrieves the current weather information for a given latitude and longitude.
3. **get_current_news**: Fetches the latest news articles based on a provided keyword.

Use these tools to gather information and provide the best possible travel advice to the user. Always be polite, clear, and concise in your responses. If you are unsure about something, let the user know and ask for clarification.

When using the tools, ensure that you provide the necessary parameters and interpret the results correctly. Always aim to provide a complete and helpful response to the user's query.
"""

# Define the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [11]:
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'question'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk